# DATA MATCHING

## Mục tiêu
- Thu thập danh sách ảnh từ 4 thư mục dữ liệu.
- Đọc và chuẩn hóa Metadata.
- Ghép ảnh với Metadata dựa trên quy tắc đánh số của GVLiD.
- Kiểm tra các bản ghi được ghép và chưa được ghép.
- Xuất dataset hoàn chỉnh dưới dạng JSON (`image_metadata.json`).

**1. IMPORT THƯ VIỆN**

In [72]:
import json
import re
import hashlib
from pathlib import Path

import pandas as pd
from IPython.display import display

**2. Kết nối dữ liệu**

Đọc Metadata từ file JSON và chuyển đổi thành DataFrame
để thuận tiện cho việc xử lý và đối chiếu.

In [73]:
# Đường dẫn đến project
PROJECT_DIR = Path(r"D:\DataMining26\Smart-Vineyard")

# Đường dẫn dataset
DATASET_DIR = (
    PROJECT_DIR
    / "Dataset"
    / "GVLiD GrapeVine Leaf identification of the Disease"
    / "DATASET"
)

# Đường dẫn metadata
METADATA_PATH = DATASET_DIR / "metadata.json"

# Thư mục lưu kết quả
OUTPUT_DIR = PROJECT_DIR / "outputs" / "processed"

# Tạo thư mục output nếu chưa tồn tại
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra đường dẫn
print("Dataset tồn tại:", DATASET_DIR.exists())
print("Metadata tồn tại:", METADATA_PATH.exists())
print("Output:", OUTPUT_DIR)

Dataset tồn tại: True
Metadata tồn tại: True
Output: D:\DataMining26\Smart-Vineyard\outputs\processed


**3. Đọc và chuẩn hóa metadata**

In [74]:
# Đọc file JSON
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

# Chuyển JSON lồng nhau thành DataFrame
df_metadata = pd.json_normalize(metadata)

print("Số bản ghi metadata:", len(df_metadata))
print("Số thuộc tính:", len(df_metadata.columns))

display(df_metadata.head())

Số bản ghi metadata: 3478
Số thuộc tính: 12


,Sr_No,Plant_Health_Status,Disease_Type,Location.Vineyard,Location.GPS.Latitude,Location.GPS.Longitude,Environmental_Conditions.Time_of_Capture,Environmental_Conditions.Lighting,Environmental_Conditions.Weather,Device_Specifications.Device_Name,Device_Specifications.Sensor,Device_Specifications.OS
0,1,Diseased,Black Rot,Vineyard_22,18.785238,73.994949,Afternoon,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
1,2,Diseased,Black Rot,Vineyard_28,18.731029,73.977206,Afternoon,Natural sunlight,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
2,3,Diseased,Black Rot,Vineyard_14,18.802297,74.027516,Afternoon,Partly cloudy,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
3,4,Diseased,Black Rot,Vineyard_20,18.775820,74.000550,Morning,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
4,5,Diseased,Black Rot,Vineyard_8,18.790580,74.002429,Evening,Partly cloudy,Clear,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android


- Đọc đúng **3.478 bản ghi × 12 thuộc tính** (Sr_No, nhãn bệnh, tình trạng sức khỏe, vườn, GPS, 3 điều kiện chụp, 3 thông tin thiết bị). JSON lồng nhau đã được làm phẳng thành các cột dạng `Location.Vineyard`, `Environmental_Conditions.Weather`...

**4. Thu thập danh sách ảnh**

In [75]:
# Ánh xạ tên thư mục với tên lớp trong metadata
class_mapping = {
    "Black rot": "Black Rot",
    "esca": "Esca",
    "healthy": "Healthy",
    "leaf blight": "Leaf Blight"
}

# Các định dạng ảnh được hỗ trợ
image_extensions = {".jpg", ".jpeg", ".png"}

image_records = []
skipped_files = []   # file bị loại do sai quy ước tên (được ghi lại để kiểm tra)

for folder_name, disease_name in class_mapping.items():

    folder_path = DATASET_DIR / folder_name

    if not folder_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục: {folder_path}"
        )

    for image_path in folder_path.iterdir():

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in image_extensions:
            continue

        # Lấy số thứ tự ở cuối tên file
        match = re.fullmatch(
            rf"{re.escape(folder_name)}(\d+)",
            image_path.stem,
            flags=re.IGNORECASE
        )

        if match is None:
            print("Tên file không đúng quy ước:", image_path.name)
            skipped_files.append(image_path)
            continue

        image_number = int(match.group(1))

        image_records.append({
            "Image_Name": image_path.name,
            "Image_Path": image_path.relative_to(DATASET_DIR).as_posix(),   # đường dẫn tương đối so với DATASET_DIR
            "Class_Folder": folder_name,
            "Disease_Type": disease_name,
            "Image_Number": image_number
        })

df_images = pd.DataFrame(image_records).sort_values(
    ["Disease_Type", "Image_Number"]
).reset_index(drop=True)

print("Tổng số ảnh:", len(df_images))

display(
    df_images.head()
)

Tên file không đúng quy ước: mega dataset (31) - Copy - Copy - Copy.jpg
Tổng số ảnh: 3476


,Image_Name,Image_Path,Class_Folder,Disease_Type,Image_Number
0,black rot001.jpg,Black rot/black rot001.jpg,Black rot,Black Rot,1
1,black rot002.jpg,Black rot/black rot002.jpg,Black rot,Black Rot,2
2,black rot003.jpg,Black rot/black rot003.jpg,Black rot,Black Rot,3
3,black rot004.jpg,Black rot/black rot004.jpg,Black rot,Black Rot,4
4,black rot005.jpg,Black rot/black rot005.jpg,Black rot,Black Rot,5


- Có **1 file sai quy ước tên** bị loại: `mega dataset (31) - Copy - Copy - Copy.jpg` (nhiều khả năng là bản sao của `black rot031.jpg`, nhưng chưa so nội dung để xác nhận).
- Còn lại **3.476 ảnh hợp lệ**.

In [76]:
print("Số ảnh theo từng lớp:")

display(
    df_images["Disease_Type"]
    .value_counts()
)

Số ảnh theo từng lớp:


Disease_Type
Healthy        1109
Esca            888
Black Rot       807
Leaf Blight     672
Name: count, dtype: int64

- Số ảnh theo lớp: Healthy 1.109, Esca 888, Black Rot 807, Leaf Blight 672.
- Black Rot ít hơn metadata 2 ảnh (807 so với 809), các lớp còn lại bằng số bản ghi metadata.

**4.1 Kiểm tra các file bị loại**

So sánh nội dung (mã MD5) của từng file bị loại với các ảnh hợp lệ cùng thư mục.
- Nếu **trùng hệt** một ảnh hợp lệ: đó là bản sao thừa, loại đi là an toàn.
- Nếu **không trùng** ảnh nào: cần mở ảnh ra xem, vì có thể đó là ảnh thật của một bản ghi đang thiếu ảnh.

In [77]:
def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


skipped_report = []
if skipped_files:
    folders = {p.parent for p in skipped_files}

    # Băm các ảnh hợp lệ nằm cùng thư mục với file bị loại
    hash_to_name = {}
    for rel in df_images["Image_Path"]:
        full = DATASET_DIR / rel
        if full.parent in folders:
            hash_to_name.setdefault(md5_of(full), rel)

    for p in skipped_files:
        same_as = hash_to_name.get(md5_of(p))
        skipped_report.append({
            "File": p.relative_to(DATASET_DIR).as_posix(),
            "Trung_voi_anh": same_as if same_as else "",
            "Ket_luan": "Bản sao thừa, loại an toàn" if same_as
                        else "KHÔNG trùng ảnh nào, cần kiểm tra thủ công",
        })

df_skipped = pd.DataFrame(
    skipped_report, columns=["File", "Trung_voi_anh", "Ket_luan"]
)

if len(df_skipped):
    display(df_skipped)
else:
    print("Không có file nào bị loại.")

,File,Trung_voi_anh,Ket_luan
0,Black rot/mega dataset (31) - Copy - Copy - Co...,Black rot/black rot001.jpg,"Bản sao thừa, loại an toàn"


**5. Đối chiếu số lượng ảnh và metadata trước khi ghép**

Trước khi ghép, so sánh số ảnh và số bản ghi của từng lớp, đồng thời xác định chính xác bản ghi nào không có ảnh (và ngược lại). Làm bước này **trước** khi ghép để quyết định loại bỏ dựa trên dữ liệu thật, không dựa vào số viết cứng.

In [78]:
n_metadata_raw = len(df_metadata)

# So sánh số lượng theo lớp
compare = pd.concat(
    [
        df_images["Disease_Type"].value_counts().rename("So_anh"),
        df_metadata["Disease_Type"].value_counts().rename("So_metadata"),
    ],
    axis=1
).fillna(0).astype(int)
compare["Chenh_lech (metadata - anh)"] = compare["So_metadata"] - compare["So_anh"]
display(compare)

# Black Rot ghép theo Sr_No: tìm bản ghi không có ảnh và ảnh không có bản ghi
black_rot_meta = df_metadata[df_metadata["Disease_Type"] == "Black Rot"]
black_rot_images = df_images[df_images["Disease_Type"] == "Black Rot"]

missing_metadata = black_rot_meta[
    ~black_rot_meta["Sr_No"].isin(black_rot_images["Image_Number"])
]
images_without_meta = black_rot_images[
    ~black_rot_images["Image_Number"].isin(black_rot_meta["Sr_No"])
]

print("Bản ghi Black Rot không có ảnh:")
display(missing_metadata[["Sr_No", "Disease_Type", "Location.Vineyard"]])
print("Số ảnh Black Rot không có bản ghi:", len(images_without_meta))

,So_anh,So_metadata,Chenh_lech (metadata - anh)
Disease_Type,,,
Healthy,1109,1109,0
Esca,888,888,0
Black Rot,807,809,2
Leaf Blight,672,672,0


Bản ghi Black Rot không có ảnh:


,Sr_No,Disease_Type,Location.Vineyard
19,20,Black Rot,Vineyard_5
808,809,Black Rot,Vineyard_16


Số ảnh Black Rot không có bản ghi: 0


-> Có thể thấy, lớp Black Rot có sự chênh lệch: 809 bản ghi metadata nhưng chỉ có 807 ảnh.
=> Hướng xử lý: loại 2 bản ghi metadata không có ảnh tương ứng (không phải xóa ảnh dư, vì không có ảnh nào dư).

**5.1 Loại các bản ghi metadata không có ảnh**

Các bản ghi không có ảnh tương ứng sẽ được loại khỏi dataset cuối. Kết quả đối chiếu trước đó cho thấy đó là **Black Rot `Sr_No` 20 và 809**. Cell dưới đây so tập bản ghi phát hiện được với kỳ vọng này: nếu khác thì **dừng lại và báo lỗi** thay vì tự loại sai.

In [79]:
EXPECTED_NO_IMAGE = {20, 809}   # Sr_No dự kiến không có ảnh

found = set(missing_metadata["Sr_No"].astype(int))
assert found == EXPECTED_NO_IMAGE, (
    f"Bản ghi không có ảnh khác kỳ vọng! Phát hiện: {sorted(found)}, kỳ vọng: {sorted(EXPECTED_NO_IMAGE)}"
)
assert len(images_without_meta) == 0, "Có ảnh Black Rot không có bản ghi metadata!"
print("✓ Bản ghi không có ảnh đúng như kỳ vọng:", sorted(found))

# Lưu lại các bản ghi bị loại để làm báo cáo, sau đó loại khỏi metadata
df_metadata_removed = df_metadata[df_metadata["Sr_No"].isin(found)].copy()
df_metadata = df_metadata[~df_metadata["Sr_No"].isin(found)].reset_index(drop=True)

print("\nTổng metadata sau khi loại:", len(df_metadata))
print(df_metadata["Disease_Type"].value_counts())

# Kiểm tra lại: không còn bản ghi Black Rot nào thiếu ảnh
still_missing = df_metadata[
    (df_metadata["Disease_Type"] == "Black Rot")
    & ~df_metadata["Sr_No"].isin(black_rot_images["Image_Number"])
]
print("\nSố bản ghi Black Rot còn thiếu ảnh:", len(still_missing))

✓ Bản ghi không có ảnh đúng như kỳ vọng: [20, 809]

Tổng metadata sau khi loại: 3476
Disease_Type
Healthy        1109
Esca            888
Black Rot       807
Leaf Blight     672
Name: count, dtype: int64

Số bản ghi Black Rot còn thiếu ảnh: 0


**6. Ghép ảnh với metadata**

In [80]:
# GHÉP ẢNH VỚI METADATA

matched_records = []

for folder_name, disease_name in class_mapping.items():

    # Lọc ảnh và metadata theo lớp
    df_class_images = df_images[
        df_images["Class_Folder"] == folder_name
    ].copy()

    df_class_metadata = df_metadata[
        df_metadata["Disease_Type"].str.strip().str.lower()
        == disease_name.lower()
    ].copy()

    # Bỏ Disease_Type từ metadata để tránh trùng cột
    df_class_metadata = df_class_metadata.drop(
        columns=["Disease_Type"]
    )

    if disease_name == "Black Rot":

        # Ghép Black Rot theo Image_Number và Sr_No
        merged = df_class_images.merge(
            df_class_metadata,
            left_on="Image_Number",
            right_on="Sr_No",
            how="inner"
        )

        # Inner join không được âm thầm bỏ ảnh hay bản ghi nào
        if not (len(merged) == len(df_class_images) == len(df_class_metadata)):
            raise ValueError(
                f"Black Rot ghép không khớp hoàn toàn: ảnh={len(df_class_images)}, "
                f"metadata={len(df_class_metadata)}, ghép được={len(merged)}"
            )

    else:

        # Sắp xếp trước khi ghép theo thứ tự
        df_class_images = df_class_images.sort_values(
            "Image_Number"
        ).reset_index(drop=True)

        df_class_metadata = df_class_metadata.sort_values(
            "Sr_No"
        ).reset_index(drop=True)

        if len(df_class_images) != len(df_class_metadata):
            raise ValueError(
                f"Số ảnh và metadata không khớp: {disease_name}"
            )

        # Ghép theo vị trí chỉ đáng tin khi số trong tên file chạy liên tục 1..n
        n_images = len(df_class_images)
        if df_class_images["Image_Number"].tolist() != list(range(1, n_images + 1)):
            raise ValueError(
                f"Số thứ tự ảnh không liên tục 1..{n_images}: {disease_name}"
            )

        merged = pd.concat(
            [
                df_class_images.reset_index(drop=True),
                df_class_metadata.reset_index(drop=True)
            ],
            axis=1
        )

    # Gán nhãn lấy từ tên folder
    merged["Disease_Type"] = disease_name

    matched_records.append(merged)

# Gộp 4 lớp
df_matched = pd.concat(
    matched_records,
    ignore_index=True
)

# Kiểm tra kết quả
print("Tổng số ảnh:", len(df_matched))
print("\nSố lượng từng lớp:")
print(df_matched["Disease_Type"].value_counts())

print("\nCác cột bị trùng:")
print(df_matched.columns[df_matched.columns.duplicated()].tolist())

display(df_matched.head())

Tổng số ảnh: 3476

Số lượng từng lớp:
Disease_Type
Healthy        1109
Esca            888
Black Rot       807
Leaf Blight     672
Name: count, dtype: int64

Các cột bị trùng:
[]


,Image_Name,Image_Path,Class_Folder,Disease_Type,Image_Number,Sr_No,Plant_Health_Status,Location.Vineyard,Location.GPS.Latitude,Location.GPS.Longitude,Environmental_Conditions.Time_of_Capture,Environmental_Conditions.Lighting,Environmental_Conditions.Weather,Device_Specifications.Device_Name,Device_Specifications.Sensor,Device_Specifications.OS
0,black rot001.jpg,Black rot/black rot001.jpg,Black rot,Black Rot,1,1,Diseased,Vineyard_22,18.785238,73.994949,Afternoon,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
1,black rot002.jpg,Black rot/black rot002.jpg,Black rot,Black Rot,2,2,Diseased,Vineyard_28,18.731029,73.977206,Afternoon,Natural sunlight,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
2,black rot003.jpg,Black rot/black rot003.jpg,Black rot,Black Rot,3,3,Diseased,Vineyard_14,18.802297,74.027516,Afternoon,Partly cloudy,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
3,black rot004.jpg,Black rot/black rot004.jpg,Black rot,Black Rot,4,4,Diseased,Vineyard_20,18.775820,74.000550,Morning,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
4,black rot005.jpg,Black rot/black rot005.jpg,Black rot,Black Rot,5,5,Diseased,Vineyard_8,18.790580,74.002429,Evening,Partly cloudy,Clear,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android


In [81]:
print(df_matched["Disease_Type"].value_counts())
print("\nSố dòng:", len(df_matched))
print("Cột trùng:", df_matched.columns[
    df_matched.columns.duplicated()
].tolist())
print("Sr_No trùng:", df_matched["Sr_No"].duplicated().sum())

Disease_Type
Healthy        1109
Esca            888
Black Rot       807
Leaf Blight     672
Name: count, dtype: int64

Số dòng: 3476
Cột trùng: []
Sr_No trùng: 0


**KIỂM TRA**

In [82]:
print("Số metadata ban đầu        :", n_metadata_raw)
print("Số bản ghi đã loại         :", len(df_metadata_removed))
print("Số metadata sau làm sạch   :", len(df_metadata))
print("Tổng số ảnh hợp lệ         :", len(df_images))
print("Tổng số bản ghi đã ghép    :", len(df_matched))

summary = pd.concat(
    [
        df_images["Disease_Type"].value_counts().rename("Anh"),
        df_metadata["Disease_Type"].value_counts().rename("Metadata (sau lam sach)"),
        df_matched["Disease_Type"].value_counts().rename("Sau khi ghep"),
    ],
    axis=1
)
display(summary)

checks = {
    "Số bản ghi ghép = số ảnh = số metadata sau làm sạch":
        len(df_matched) == len(df_images) == len(df_metadata),
    "Số lượng từng lớp khớp giữa ảnh, metadata và sau ghép":
        bool((summary.nunique(axis=1) == 1).all()),
    "Sr_No không trùng": not df_matched["Sr_No"].duplicated().any(),
    "Không có cột trùng": not df_matched.columns.duplicated().any(),
    "Nhãn bệnh và tình trạng sức khỏe nhất quán": bool(
        ((df_matched["Disease_Type"] == "Healthy")
         == (df_matched["Plant_Health_Status"] == "Healthy")).all()
    ),
    "Mọi đường dẫn ảnh đều tồn tại":
        all((DATASET_DIR / p).exists() for p in df_matched["Image_Path"]),
}
print()
for name, ok in checks.items():
    print("✓" if ok else "✗", name)

ALL_OK = all(checks.values())
print("\nKết luận:", "ĐẠT, có thể xuất dữ liệu." if ALL_OK else "CHƯA ĐẠT, cần kiểm tra lại các mục ✗ ở trên.")

Số metadata ban đầu        : 3478
Số bản ghi đã loại         : 2
Số metadata sau làm sạch   : 3476
Tổng số ảnh hợp lệ         : 3476
Tổng số bản ghi đã ghép    : 3476


,Anh,Metadata (sau lam sach),Sau khi ghep
Disease_Type,,,
Healthy,1109,1109,1109
Esca,888,888,888
Black Rot,807,807,807
Leaf Blight,672,672,672



✓ Số bản ghi ghép = số ảnh = số metadata sau làm sạch
✓ Số lượng từng lớp khớp giữa ảnh, metadata và sau ghép
✓ Sr_No không trùng
✓ Không có cột trùng
✓ Nhãn bệnh và tình trạng sức khỏe nhất quán
✓ Mọi đường dẫn ảnh đều tồn tại

Kết luận: ĐẠT, có thể xuất dữ liệu.


**7. Xuất dữ liệu đã ghép**

In [83]:
if not ALL_OK:
    raise RuntimeError("Dữ liệu chưa đạt điều kiện kiểm tra, KHÔNG xuất file!")

import json

output_file = OUTPUT_DIR / "image_metadata.json"

df_matched.to_json(
    output_file,
    orient="records",
    force_ascii=False,
    indent=4
)

print("Đã lưu dữ liệu tại:")
print(output_file)

print("Tổng số bản ghi:", len(df_matched))

# File báo cáo kèm theo: bản ghi bị loại và file ảnh bị loại
df_metadata_removed.to_csv(
    OUTPUT_DIR / "metadata_without_image.csv", index=False, encoding="utf-8-sig"
)
df_skipped.to_csv(
    OUTPUT_DIR / "skipped_files.csv", index=False, encoding="utf-8-sig"
)
print("Đã lưu báo cáo:", "metadata_without_image.csv,", "skipped_files.csv")

Đã lưu dữ liệu tại:
D:\DataMining26\Smart-Vineyard\outputs\processed\image_metadata.json
Tổng số bản ghi: 3476
Đã lưu báo cáo: metadata_without_image.csv, skipped_files.csv


In [84]:
with open(output_file, "r", encoding="utf-8") as f:
    check_data = json.load(f)

print("Số bản ghi:", len(check_data))
print("Các trường dữ liệu:", list(check_data[0].keys()))
print("Mọi đường dẫn ảnh đều tồn tại:",
      all((DATASET_DIR / rec["Image_Path"]).exists() for rec in check_data))

Số bản ghi: 3476
Các trường dữ liệu: ['Image_Name', 'Image_Path', 'Class_Folder', 'Disease_Type', 'Image_Number', 'Sr_No', 'Plant_Health_Status', 'Location.Vineyard', 'Location.GPS.Latitude', 'Location.GPS.Longitude', 'Environmental_Conditions.Time_of_Capture', 'Environmental_Conditions.Lighting', 'Environmental_Conditions.Weather', 'Device_Specifications.Device_Name', 'Device_Specifications.Sensor', 'Device_Specifications.OS']
Mọi đường dẫn ảnh đều tồn tại: True
